# NYC Taxi Dataset Spatial Joins with H3 Turbo

This notebook demonstrates how to create a dataset resembling NYC Taxi pickups, convert the coordinates to H3 cells using the high-performance `h3_turbo` library, and perform rapid spatial joins against target zones.

In [ ]:
import os
import sys
import glob

# Ensure h3_turbo is available in the environment
venv_patterns = [
    '/opt/venv/lib/python*/site-packages',
    '/app/.venv*/lib/python*/site-packages',
    '/workspace/.venv*/lib/python*/site-packages',
    os.path.join(os.getcwd(), '.venv312', 'lib', 'python*', 'site-packages')
]
found_paths = []
for pattern in venv_patterns:
    found_paths.extend(glob.glob(pattern))

for p in found_paths:
    if p not in sys.path:
        sys.path.append(p)

import h3_turbo
import numpy as np
import pandas as pd
import time

print('Libraries imported successfully.')

## 1. Generate NYC Taxi Style Table
We generate 10 million random coordinates corresponding roughly to the bounding box of NYC.

In [ ]:
def get_nyc_taxi_data(n_points=10_000_000):
    """
    Generate dummy NYC taxi dataset.
    NYC boundaries roughly: Lat 40.5 to 40.9, Lng -74.25 to -73.7
    """
    print(f"Generating {n_points:,} NYC taxi pickup points...")
    lats = np.random.uniform(40.5, 40.9, n_points).astype(np.float64)
    lngs = np.random.uniform(-74.25, -73.7, n_points).astype(np.float64)
    return lats, lngs

n_points = 10_000_000
lats, lngs = get_nyc_taxi_data(n_points)

# Display as a pandas DataFrame to represent a table
df = pd.DataFrame({'pickup_latitude': lats, 'pickup_longitude': lngs})
print("\nSample of the NYC Taxi table:")
display(df.head())

## 2. Fast Lat/Lng to H3 Conversion
We use `h3_turbo.latlons_to_h3s` to convert the 10 million latitude and longitude coordinates to H3 cells at resolution 9.

In [ ]:
res = 9  # H3 Resolution for taxi zones (~174 meters edge length)

print(f"Converting {n_points:,} Lat/Lng pairs to H3 cells at resolution {res}...")

# Warmup the JIT compiler for accurate benchmarking
h3_turbo.warmup()

start_time = time.time()
# High-performance batch conversion on GPU/Accelerators
h3_cells = h3_turbo.latlons_to_h3s(lats, lngs, res)
end_time = time.time()

elapsed = end_time - start_time
print(f"Conversion took {elapsed:.4f} seconds.")
print(f"Throughput: {n_points / elapsed:,.0f} conversions/second.")
print(f"Sample converted H3 cells (Hex): {[hex(x) for x in h3_cells[:5]]}")

# Add H3 cells to the table (stored as uint64)
df['pickup_h3'] = h3_cells

## 3. Generate Target Zones
We generate a set of target zones in the same area to simulate specific neighborhoods or high-demand taxi zones.

In [ ]:
n_zones = 5_000
print(f"Generating {n_zones:,} target zones...")
zone_lats = np.random.uniform(40.5, 40.9, n_zones).astype(np.float64)
zone_lngs = np.random.uniform(-74.25, -73.7, n_zones).astype(np.float64)

zone_cells = h3_turbo.latlons_to_h3s(zone_lats, zone_lngs, res)
# Ensure zones are unique for the spatial join
zone_cells = np.unique(zone_cells)

print(f"Generated {len(zone_cells):,} unique target H3 zones.")

## 4. Perform Spatial Join
We use `h3_turbo.spatial_join` to test which of our 10 million pickups fall inside any of our target zones. This simulates a high-throughput Point-in-Polygon spatial join operation.

In [ ]:
print(f"Performing Spatial Join: {n_points:,} pickups against {len(zone_cells):,} zones...")

start_time = time.time()
# Perform high-speed spatial join
# Returns a boolean mask (or uint8 mask) of points that fall within any of the provided zones
is_in_zone = h3_turbo.spatial_join(h3_cells, zone_cells, res)
end_time = time.time()

elapsed_gpu = end_time - start_time
print(f"Spatial join took {elapsed_gpu:.4f} seconds.")
print(f"Throughput: {n_points / elapsed_gpu:,.0f} points/second.")

points_in_zones = np.sum(is_in_zone)
print(f"Found {points_in_zones:,} points inside the target zones.")
print(f"Percentage of pickups in target zones: {(points_in_zones / n_points) * 100:.2f}%")

# Add the result mask to our table
df['in_target_zone'] = is_in_zone
print("\nFinal Dataset with Spatial Join Results:")
display(df.head(10))

## 5. CPU Baseline (Multi-threaded)
For comparison, we perform the same logic using standard Python multi-processing and NumPy on the CPU.

In [ ]:
import platform
import concurrent.futures
import multiprocessing
import os

try:
    num_workers = len(os.sched_getaffinity(0))
except AttributeError:
    num_workers = os.cpu_count() or 1

print(f"Running CPU baseline on {platform.processor()} with {num_workers} vCPUs...")

# We use a global set for fast lookups across workers
_global_zone_set = None

def _init_worker(zone_set):
    global _global_zone_set
    _global_zone_set = zone_set

def _cpu_spatial_join_chunk(chunk):
    return np.array([1 if p in _global_zone_set else 0 for p in chunk], dtype=np.uint8)

zone_set = set(zone_cells)

start_cpu = time.time()
chunk_size = 1_000_000
chunks = [h3_cells[i:i + chunk_size] for i in range(0, len(h3_cells), chunk_size)]

with concurrent.futures.ProcessPoolExecutor(max_workers=num_workers, initializer=_init_worker, initargs=(zone_set,)) as executor:
    results = list(executor.map(_cpu_spatial_join_chunk, chunks))

cpu_is_in_zone = np.concatenate(results)
cpu_elapsed = time.time() - start_cpu

print(f"CPU spatial join took {cpu_elapsed:.4f} seconds.")
print(f"CPU Throughput: {n_points / cpu_elapsed:,.0f} points/second.")
if elapsed_gpu > 0:
    print(f"Speedup (GPU vs CPU): {cpu_elapsed / elapsed_gpu:.2f}x")

assert np.array_equal(is_in_zone, cpu_is_in_zone), "CPU and GPU results do not match!"
print("CPU Baseline Verification Passed.")